In [9]:
# 関数の定義
# CAM部
def calc_cam_oprations(
        neuron_num: int,
        input_dim: int,
        cam_bit: int,
        odeu: int
):
    entries = 2 ** cam_bit
    # ビット数
    cam_bits = entries * cam_bit
    # 演算回数
    cam_oprations = odeu * (neuron_num + input_dim) * entries

    return cam_bits, cam_oprations

# LUT部
def calc_lut_oprations(
        neuron_num: int,
        input_dim: int,
        cam_bit: int,
        lut_bit: int,
        odeu: int
):
    entries = 2 ** cam_bit
    # ビット数
    lut_bits = entries * neuron_num * lut_bit

    # 演算回数
    lut_oprations = odeu * (neuron_num + input_dim) * neuron_num

    return lut_bits, lut_oprations

# MVM部
def calc_mvm_oprations(
        neuron_num: int,
        input_dim: int,
        mvm_bit: int,
        odeu: int
):
    # ビット数
    mvm_bits = neuron_num * (neuron_num + input_dim) * mvm_bit

    # 演算回数 (Add/Mul)
    mvm_oprations = 2 * odeu * (neuron_num + input_dim) * neuron_num

    return mvm_bits, mvm_oprations

# Arithmetic部
def calc_arithmetic_oprations(
        neuron_num: int,
        arithmetic_bit_RRAM: int,
        arithmetic_bit_SRAM: int,
        odeu: int
):
    # ビット数 (保存用)
    arithmetic_bits = 3 * neuron_num * arithmetic_bit_RRAM + neuron_num * arithmetic_bit_SRAM

    # 演算回数 (Add/Mul/Div)
    arithmetic_oprations = 6 * odeu * neuron_num

    return arithmetic_bits, arithmetic_oprations


In [11]:
# 全体のビット数と演算量の計算と可視化
def calc_total_oprations(
        neuron_num: int,
        input_dim: int,
        cam_bit: int,
        lut_bit: int,
        mvm_bit: int,
        arithmetic_bit_RRAM: int,
        arithmetic_bit_SRAM: int,
        odeu: int
):
    cam_bits, cam_oprations = calc_cam_oprations(
        neuron_num,
        input_dim,
        cam_bit,
        odeu
    )

    lut_bits, lut_oprations = calc_lut_oprations(
        neuron_num,
        input_dim,
        cam_bit,
        lut_bit,
        odeu
    )

    mvm_bits, mvm_oprations = calc_mvm_oprations(
        neuron_num,
        input_dim,
        mvm_bit,
        odeu
    )

    arithmetic_bits, arithmetic_oprations = calc_arithmetic_oprations(
        neuron_num,
        arithmetic_bit_RRAM,
        arithmetic_bit_SRAM,
        odeu
    )

    total_bits = cam_bits + lut_bits + mvm_bits + arithmetic_bits
    total_oprations = cam_oprations + lut_oprations + mvm_oprations + arithmetic_oprations

    return {
        "CAM": {"bits": cam_bits, "operations": cam_oprations},
        "LUT": {"bits": lut_bits, "operations": lut_oprations},
        "MVM": {"bits": mvm_bits, "operations": mvm_oprations},
        "Arithmetic": {"bits": arithmetic_bits, "operations": arithmetic_oprations},
        "Total": {"bits": total_bits, "operations": total_oprations}
    }

import pandas as pd

res = calc_total_oprations(
    neuron_num=256,
    input_dim=19,
    cam_bit=8,
    lut_bit=8,
    mvm_bit=8,
    arithmetic_bit_RRAM=8,
    arithmetic_bit_SRAM=8,
    odeu=3
)

rows = []
for k in ["CAM", "LUT", "MVM", "Arithmetic"]:
    rows.append({
        "Module": k,
        "Bits": res[k]["bits"],
        "Bit Ratio [%]": 100 * res[k]["bits"] / res["Total"]["bits"],
        "Operations": res[k]["operations"],
        "Op Ratio [%]": 100 * res[k]["operations"] / res["Total"]["operations"],
    })

df = pd.DataFrame(rows)

# Total 行を追加
df.loc[len(df)] = {
    "Module": "Total",
    "Bits": res["Total"]["bits"],
    "Bit Ratio [%]": 100.0,
    "Operations": res["Total"]["operations"],
    "Op Ratio [%]": 100.0,
}

df


,Module,Bits,Bit Ratio [%],Operations,Op Ratio [%]
0,CAM,2048,0.186567,211200,24.864376
1,LUT,524288,47.761194,211200,24.864376
2,MVM,563200,51.305970,422400,49.728752
3,Arithmetic,8192,0.746269,4608,0.542495
4,Total,1097728,100.000000,849408,100.000000
